## **모델 훈련 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch04 연습문제 1, 5, 9, 10
- 개념 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

### **1. 수백만 개의 특성을 가진 훈련 세트에서는 어떤 선형 회귀 알고리즘을 사용할 수 있을까요?**
___


특성이 너무 많으면 정규 방정식이나 SVD 분해 방식은 계산 복잡도가 특성 수의 제곱에 비례하여 급격히 느려집니다. 반면, 경사 하강법은 특성 수에 민감하지 않아 대규모 데이터셋에 적합합니다. 따라서 확률적 경사 하강법이나 미니배치 경사 하강법을 사용해야 합니다.

### **2. 배치 경사 하강법을 사용하고 에포크마다 검증 오차를 그래프로 나타내봤습니다. 검증 오차가 일정하게 상승되고 있다면 어떤 일이 일어나고 있는 걸까요? 이 문제를 어떻게 해결할 수 있나요?**
___

모델이 훈련 데이터에 너무 맞춰지면서 일반화 성능이 떨어지는 과대적합이 발생하고 있거나, 학습률이 너무 커서 최솟값에서 발산하고 있는 상태입니다. 과대적합일 경우 조기 종료를 사용하여 검증 오차가 최소에 도달했을 때 훈련을 멈춥니다. 발산할 경우 학습률을 낮춰서 가중치가 점진적으로 수렴하게 만듭니다.

### **3. 릿지 회귀를 사용했을 때 훈련 오차가 검증 오차가 거의 비슷하고 둘 다 높았습니다. 이 모델에는 높은 편향이 문제인가요, 아니면 높은 분산이 문제인가요? 규제 하이퍼파라미터 $\alpha$를 증가시켜야 할까요 아니면 줄여야 할까요?**
___

높은 편향이 문제입니다. 규제 하이퍼파라미터 alpha를 줄여야 합니다.

### **4. 다음과 같이 사용해야 하는 이유는?**
___
- 평범한 선형 회귀(즉, 아무런 규제가 없는 모델) 대신 릿지 회귀
- 릿지 회귀 대신 라쏘 회귀
- 라쏘 회귀 대신 엘라스틱넷

평범한 선형 회귀 대신 릿지: 규제가 없는 모델보다 약간의 규제가 있는 모델이 일반화 성능이 더 좋기 때문입니다.
릿지 대신 라쏘: 실제로 중요한 특성이 몇 개뿐이라고 의심될 때 사용합니다. 라쏘는 덜 중요한 특성의 가중치를 0으로 만들어 자동으로 특성을 선택해 줍니다.
라쏘 대신 엘라스틱넷: 특성 수가 훈련 샘플 수보다 많거나, 여러 특성이 강하게 연관(상관관계)되어 있을 때 라쏘보다 안정적으로 작동합니다.

### **추가) 조기 종료를 사용한 배치 경사 하강법으로 iris 데이터를 활용해 소프트맥스 회귀를 구현해보세요(사이킷런은 사용하지 마세요)**


---



In [4]:
import numpy as np
from sklearn.datasets import load_iris

# 1. 데이터 로드 및 전처리
iris = load_iris()
X = iris["data"]
y = iris["target"]

# 편향(bias)을 위한 특성 추가 (X_with_bias 정의)
X_with_bias = np.c_[np.ones([len(X), 1]), X]

# 데이터 분할 (훈련/검증/테스트 세트)
test_ratio = 0.2
validation_ratio = 0.2
total_size = len(X_with_bias)

test_size = int(total_size * test_ratio)
validation_size = int(total_size * validation_ratio)
train_size = total_size - test_size - validation_size

rnd_indices = np.random.permutation(total_size)

X_train = X_with_bias[rnd_indices[:train_size]]
y_train = y[rnd_indices[:train_size]]
X_valid = X_with_bias[rnd_indices[train_size:-test_size]]
y_valid = y[rnd_indices[train_size:-test_size]]
X_test = X_with_bias[rnd_indices[-test_size:]]
y_test = y[rnd_indices[-test_size:]]

# 타겟 레이블 원-핫 인코딩 함수
def to_one_hot(y):
    n_classes = y.max() + 1
    m = len(y)
    Y_one_hot = np.zeros((m, n_classes))
    Y_one_hot[np.arange(m), y] = 1
    return Y_one_hot

y_train_one_hot = to_one_hot(y_train)
y_valid_one_hot = to_one_hot(y_valid)
y_test_one_hot = to_one_hot(y_test)

# 2. 필수 함수 정의 (Softmax & Loss)
def softmax(logits):
    exps = np.exp(logits)
    exp_sums = np.sum(exps, axis=1, keepdims=True)
    return exps / exp_sums

# 3. 모델 훈련 설정
eta = 0.1
n_iterations = 5001
best_loss = np.inf
epsilon = 1e-7 # 로그 계산 시 무한대 방지

# 가중치 초기화 (오류가 났던 지점)
Theta = np.random.randn(X_with_bias.shape[1], len(np.unique(y)))

# 4. 훈련 루프
for iteration in range(n_iterations):
    # 소프트맥스 점수 및 확률 계산
    logits = X_train.dot(Theta)
    Y_proba = softmax(logits)
    
    # 오차 및 그레디언트 계산
    error = Y_proba - y_train_one_hot
    gradients = 1/len(X_train) * X_train.T.dot(error)
    
    # 가중치 업데이트
    Theta = Theta - eta * gradients
    
    # 조기 종료 검증
    logits_val = X_valid.dot(Theta)
    Y_proba_val = softmax(logits_val)
    # 크로스 엔트로피 손실 계산
    loss_val = -np.mean(np.sum(y_valid_one_hot * np.log(Y_proba_val + epsilon), axis=1))
    
    if loss_val < best_loss:
        best_loss = loss_val
    else:
        print(f"{iteration}회에서 조기 종료! (최저 손실: {best_loss:.4f})")
        break

# 5. 테스트 세트 평가
logits_test = X_test.dot(Theta)
Y_proba_test = softmax(logits_test)
y_predict = np.argmax(Y_proba_test, axis=1)
accuracy = np.mean(y_predict == y_test)
print(f"최종 테스트 정확도: {accuracy:.2%}")

2회에서 조기 종료! (최저 손실: 1.0539)
최종 테스트 정확도: 63.33%
